# Google Gemini API — Notebook Panduan Cepat (Interactions API)

Dokumentasi resmi: **https://aistudio.google.com/docs/text-generation**

Notebook langsung-pakai untuk memanggil **Google Gemini API** dari Python
menggunakan SDK **`google-genai`** modern dengan API **Interactions** yang baru
(`client.interactions.create(...)`).

## Yang akan kamu lakukan di sini
1. Instal dependensi
2. Menyiapkan API key
3. Membuat klien
4. Generasi teks dasar
5. *Thinking* dengan Gemini
6. System instruction & konfigurasi
7. Streaming respons
8. Percakapan multi-giliran

## Prasyarat
- API key Google AI Studio: https://aistudio.google.com/apikey
- Python 3.9+

> Simpan key kamu di environment variable bernama `GOOGLE_API_KEY` (atau
> `GEMINI_API_KEY`). Lihat sel penyiapan di bawah.


## 1. Install dependensi

Jalankan sel berikut satu kali. Sel ini menginstal SDK Google GenAI resmi.


In [ ]:
# Instal SDK Google GenAI resmi (klien Gemini modern)
!pip install -q google-genai

# Verifikasi impor berhasil
from google import genai
print("google-genai berhasil diimpor:", genai.__version__ if hasattr(genai, "__version__") else "versi tidak tersedia")


## 2. Siapkan API key kamu

SDK membaca key dari `GOOGLE_API_KEY` secara default. Set sekali di shell kamu
(bukan di notebook, supaya tidak menyimpan rahasia di file):

```bash
export GOOGLE_API_KEY="isi-api-key-kamu"
```

Atau tambahkan ke file `.env` lalu muat di sini. Sel ini lebih mengutamakan
environment variable dan baru memakai variabel teks polos **hanya sebagai jalan
terakhir** — environment variable adalah jalur yang paling aman.


In [ ]:
import os

# 1) Utamakan environment variable
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 2) Opsional: muat dari file .env lokal
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
    except ImportError:
        pass

# 3) Jalan terakhir HANYA untuk demo cepat — gunakan env var di produksi
if not api_key:
    api_key = "ISI_API_KEY_KAMU_DI_SINI"

if api_key == "ISI_API_KEY_KAMU_DI_SINI" or not api_key:
    print("PERINGATAN: API key tidak ditemukan. Set GOOGLE_API_KEY di lingkungan kamu.")
else:
    print("API key dimuat ({} karakter)".format(len(api_key)))


## 3. Buat klien

Model `gemini-3.5-flash-lite` dipakai secara default. Ganti nama model sesuai kebutuhan kamu.


In [ ]:
from google import genai

# Klien otomatis mengambil GOOGLE_API_KEY dari environment jika ada.
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"
print("Klien siap:", client)


## 4. Generasi teks dasar

Panggilan paling sederhana: prompt masuk, teks keluar. Gunakan properti
`interaction.output_text` untuk mengakses teks terakhir dari respons model.


In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input="Bagaimana cara kerja AI?"
)

print(interaction.output_text)


## 5. Thinking dengan Gemini

Model Gemini sering punya "thinking" aktif secara default, yang memungkinkan
model bernalar sebelum merespons. Atur level thinking dengan
`generation_config={"thinking_level": "low"}` untuk mengontrol biaya, latensi,
dan kecerdasan.


In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input="Ada berapa banyak planet di tata surya kita?",
    generation_config={
        "thinking_level": "low"
    }
)

thought_text = []
for step in interaction.steps:
    if step.type == "thought":
        for block in (step.summary or []):
            if block.type == "text":
                thought_text.append(block.text)

print("Langkah-langkah pemikiran:")
print("\n".join(thought_text))

print("Jawaban:", interaction.output_text)
# print(interaction.output_text)


In [ ]:
interaction = client.interactions.create(
    model=MODEL,                                   
    input="What is the sum of the first 50 prime numbers?",
    generation_config={
        "thinking_level": "low",        # (minimal/low/medium/high)
        "thinking_summaries": "auto",   # (hanya auto/none)
    }
)

for step in interaction.steps:
    if step.type == "thought":
        print("Thought summary:")
        if step.summary:
            for block in step.summary:
                if block.type == "text":
                    print(block.text)
        print()
    elif step.type == "model_output":
        for block in step.content:
            if block.type == "text":
                print("Answer:")
                print(block.text)
                print()

## 6. System instruction & konfigurasi lainnya

Pandu perilaku model dengan `system_instruction`, lalu sesuaikan parameter
generasi (misalnya `temperature`) lewat `generation_config`.


In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    system_instruction="Kamu adalah seorang bajak laut. Namamu Luffy.",
    input="Halo",
)

print("Jawaban Luffy:", interaction.output_text)

In [ ]:
# Contoh lain: menimpa suhu (temperature) default
interaction2 = client.interactions.create(
    model=MODEL,
    input="Jelaskan cara kerja AI",
    generation_config={
        "temperature": 1.0
    }
)
print("Jawaban dengan suhu 1.0:", interaction2.output_text)


## 7. Streaming respons

Secara default model mengembalikan respons setelah proses generasi selesai.
Untuk interaksi yang lebih mengalir, gunakan streaming dengan `stream=True`
dan baca setiap *event* yang dikirim bertahap.


In [ ]:
stream = client.interactions.create(
    model=MODEL,
    input="Jelaskan cara kerja AI",
    stream=True,
)

for event in stream:
    if event.event_type == "step.delta":
        if event.delta.type == "text":
            print(event.delta.text, end="")

## Selanjutnya

- Coba Gemini di Google AI Studio.
- Bereksperimen dengan output terstruktur untuk respons seperti JSON.
- Pelajari strategi *prompting*.

> Dokumentasi resmi: pembuatan teks pada API Interactions.
    https://aistudio.google.com/docs/text-generation
